# Chapter 36 — Capstone planning: sales analytics workbench requirements and data model

**Book role:** Capstone  
**Suggested book space:** about 8 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Translate an analytics brief into acceptance criteria.
- Define the sales dataset grain, data dictionary, and source contracts.
- Plan views, filters, error states, and integration seams.
- Prioritize an MVP before optional database connectors.

## 1. Define the product and its users

The capstone is a desktop Sales Analytics Workbench for a student analyst or small operations team. Its primary job is to import a sales dataset, validate it, filter records, summarize performance, explore trends, and export a reproducible report. It is not an unrestricted database administration tool and should not expose raw query execution to ordinary users.

Write acceptance criteria as observable outcomes: the application can open CSV and Excel; it explains missing required columns; the user can filter by date/region/category/status; KPIs reconcile with a defined calculation; table and chart use the same filters; the user can export the current filtered data and a summary; and an unavailable database connector does not disable local-file functionality.

### Worked example 1: Define explicit capstone acceptance criteria

Turn each item into a testable acceptance statement and reference it from the project README.

In [ ]:
CAPSTONE_ACCEPTANCE = {
    "imports": ["CSV", "Excel selected sheet"],
    "validation": ["required columns", "invalid dates", "missing values", "duplicate-key review"],
    "filters": ["date range", "region", "category", "status"],
    "kpis": ["revenue", "profit", "margin", "unique orders", "unique customers"],
    "views": ["bounded table", "monthly trend", "data-quality summary"],
    "exports": ["filtered CSV", "multi-sheet Excel report"],
    "quality": ["tests", "logging", "safe error handling", "no hard-coded secrets"],
}
for area, criteria in CAPSTONE_ACCEPTANCE.items():
    print(f"{area.title()}: {', '.join(criteria)}")

## 2. Specify the canonical sales schema

The course sample models one row per order line. That grain matters: `order_id` may repeat, and order counts must use `nunique` rather than line count. A canonical schema includes order ID, order date, customer ID, region, product, category, quantity, unit price, unit cost, and status. Derived measures include revenue = quantity × unit price, cost = quantity × unit cost, and profit = revenue − cost. Margin is profit/revenue when revenue is nonzero.

Specify dtypes, allowed statuses, nullability, currency, timezone, rounding, treatment of cancelled/returned orders, and required source columns. The same schema should be used by CSV/Excel readers and database adapters, or the adapter must provide a documented mapping. Avoid mixing a line-level total with an order-level fact without aggregation rules.

### Worked example 2: Write the data dictionary

Use this as the initial schema specification. Align actual handling of status/return rows with these definitions and update the dictionary when rules change.

In [ ]:
DATA_DICTIONARY = [
    ("order_id", "string", "Order identifier; may repeat because data is line-level"),
    ("order_date", "datetime", "Business date/time; timezone policy must be documented"),
    ("customer_id", "string", "Customer identifier; not an arithmetic measure"),
    ("region", "category", "Sales region from the allowed category list"),
    ("quantity", "integer", "Units on this order line; negative values require a return rule"),
    ("unit_price", "decimal/numeric", "Selling price per unit in the declared currency"),
    ("unit_cost", "decimal/numeric", "Cost per unit; missing cost is not automatically zero"),
    ("status", "category", "Order status; cancelled/returned treatment is explicit"),
    ("revenue", "derived numeric", "quantity × unit_price under the chosen sign/return policy"),
    ("profit", "derived numeric", "quantity × (unit_price - unit_cost)"),
]
for name, dtype, definition in DATA_DICTIONARY:
    print(f"{name:14} | {dtype:18} | {definition}")

## 3. Architecture and delivery phases

Begin with an MVP: load CSV, profile/validate, filter, display a bounded table, show KPIs, chart monthly revenue, and export results. Then add Excel multi-sheet export and optional MySQL/MongoDB adapters. The data layer should expose an agreed schema; a service layer should compute metrics; views should render results. The GUI should not contain database credentials or SQL query-building logic.

Plan a milestone for tests and usability before styling. A visually polished interface that double-counts order value is not a successful analytics tool. Keep a known small fixture with manually verified metrics as an acceptance reference. Add a changelog and versioned data dictionary so your book can show how an initially small project becomes a maintainable application.

## Guided lab: Lab: capstone design review

Produce a system diagram, data dictionary, acceptance-test matrix, screen sketch, error-state matrix, and milestone plan. State the row grain and metric definitions. Identify which features are required for the MVP and which depend on MySQL/MongoDB server availability. Have another student review the plan before implementation.

## Review questions and practice

1. What is the row grain of the course sales dataset?
2. Why should optional database integrations follow the local-file MVP?
3. Which metrics may be wrong if line count is mistaken for order count?
4. What makes an acceptance criterion testable?

## Further reading

- Project structure: see the course README and `data_dictionary.md` in the course pack

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.